## Example 1: WMS capabilities parsed from an embedded document

In [ ]:
# Example 1: WMS (Web Map Service) client on an embedded GetCapabilities document.
# Passing xml= makes owslib parse the document instead of fetching it, so no network is needed.
from owslib.wms import WebMapService

WMS_CAPABILITIES = b'''<?xml version="1.0" encoding="UTF-8"?>
<WMS_Capabilities version="1.3.0" xmlns="http://www.opengis.net/wms"
    xmlns:xlink="http://www.w3.org/1999/xlink">
  <Service>
    <Name>WMS</Name>
    <Title>Example Weather Map Service</Title>
    <Abstract>Offline capabilities document for a smoke test.</Abstract>
    <KeywordList><Keyword>weather</Keyword><Keyword>radar</Keyword></KeywordList>
    <OnlineResource xlink:type="simple" xlink:href="https://example.com/wms"/>
  </Service>
  <Capability>
    <Request>
      <GetCapabilities>
        <Format>text/xml</Format>
        <DCPType><HTTP><Get><OnlineResource xlink:type="simple" xlink:href="https://example.com/wms?"/></Get></HTTP></DCPType>
      </GetCapabilities>
      <GetMap>
        <Format>image/png</Format>
        <Format>image/jpeg</Format>
        <DCPType><HTTP><Get><OnlineResource xlink:type="simple" xlink:href="https://example.com/wms?"/></Get></HTTP></DCPType>
      </GetMap>
    </Request>
    <Exception><Format>XML</Format></Exception>
    <Layer>
      <Title>Example layers</Title>
      <CRS>EPSG:4326</CRS>
      <CRS>EPSG:3857</CRS>
      <EX_GeographicBoundingBox>
        <westBoundLongitude>-180</westBoundLongitude>
        <eastBoundLongitude>180</eastBoundLongitude>
        <southBoundLatitude>-90</southBoundLatitude>
        <northBoundLatitude>90</northBoundLatitude>
      </EX_GeographicBoundingBox>
      <Layer queryable="1">
        <Name>radar</Name>
        <Title>Weather Radar</Title>
        <Abstract>Composite radar reflectivity.</Abstract>
        <EX_GeographicBoundingBox>
          <westBoundLongitude>-125</westBoundLongitude>
          <eastBoundLongitude>-66</eastBoundLongitude>
          <southBoundLatitude>25</southBoundLatitude>
          <northBoundLatitude>49</northBoundLatitude>
        </EX_GeographicBoundingBox>
        <Style><Name>default</Name><Title>Default</Title></Style>
      </Layer>
      <Layer queryable="0">
        <Name>temperature</Name>
        <Title>Surface Temperature</Title>
      </Layer>
    </Layer>
  </Capability>
</WMS_Capabilities>
'''

wms = WebMapService('https://example.com/wms', version='1.3.0', xml=WMS_CAPABILITIES)
print(f"WMS version: {wms.version}")
print(f"Title: {wms.identification.title}")
print(f"Keywords: {wms.identification.keywords}")
print(f"Layers: {list(wms.contents)}")

assert wms.version == '1.3.0'
assert wms.identification.title == 'Example Weather Map Service'
assert wms.identification.keywords == ['weather', 'radar']
assert list(wms.contents) == ['radar', 'temperature']

radar = wms['radar']
print(f"radar: title={radar.title!r}, bbox={radar.boundingBoxWGS84}, crs={radar.crsOptions}")
assert radar.title == 'Weather Radar'
assert radar.boundingBoxWGS84 == (-125.0, 25.0, -66.0, 49.0)
assert set(radar.crsOptions) == {'EPSG:4326', 'EPSG:3857'}  # inherited from the parent layer
assert 'default' in radar.styles
assert int(radar.queryable) == 1 and int(wms['temperature'].queryable) == 0

get_map = wms.getOperationByName('GetMap')
print(f"GetMap formats: {get_map.formatOptions}")
assert get_map.formatOptions == ['image/png', 'image/jpeg']
assert [op.name for op in wms.operations] == ['GetCapabilities', 'GetMap']

print("Example 1 finished")

## Example 2: WFS capabilities and a GetFeature request URL

In [ ]:
# Example 2: WFS (Web Feature Service) client on an embedded 2.0.0 GetCapabilities document
from urllib.parse import parse_qs, urlparse

from owslib.wfs import WebFeatureService

WFS_CAPABILITIES = b'''<?xml version="1.0" encoding="UTF-8"?>
<wfs:WFS_Capabilities version="2.0.0"
    xmlns:wfs="http://www.opengis.net/wfs/2.0"
    xmlns:ows="http://www.opengis.net/ows/1.1"
    xmlns:xlink="http://www.w3.org/1999/xlink">
  <ows:ServiceIdentification>
    <ows:Title>Example Hydrology Feature Service</ows:Title>
    <ows:Abstract>Offline capabilities document for a smoke test.</ows:Abstract>
    <ows:Keywords><ows:Keyword>water</ows:Keyword><ows:Keyword>monitoring</ows:Keyword></ows:Keywords>
    <ows:ServiceType>WFS</ows:ServiceType>
    <ows:ServiceTypeVersion>2.0.0</ows:ServiceTypeVersion>
  </ows:ServiceIdentification>
  <ows:OperationsMetadata>
    <ows:Operation name="GetCapabilities">
      <ows:DCP><ows:HTTP><ows:Get xlink:href="https://example.com/wfs?"/></ows:HTTP></ows:DCP>
    </ows:Operation>
    <ows:Operation name="DescribeFeatureType">
      <ows:DCP><ows:HTTP><ows:Get xlink:href="https://example.com/wfs?"/></ows:HTTP></ows:DCP>
    </ows:Operation>
    <ows:Operation name="GetFeature">
      <ows:DCP><ows:HTTP><ows:Get xlink:href="https://example.com/wfs?"/><ows:Post xlink:href="https://example.com/wfs"/></ows:HTTP></ows:DCP>
    </ows:Operation>
  </ows:OperationsMetadata>
  <wfs:FeatureTypeList>
    <wfs:FeatureType>
      <wfs:Name>hydro:water_stations</wfs:Name>
      <wfs:Title>Water Monitoring Stations</wfs:Title>
      <wfs:Abstract>Stream gauges.</wfs:Abstract>
      <wfs:DefaultCRS>urn:ogc:def:crs:EPSG::4326</wfs:DefaultCRS>
      <wfs:OtherCRS>urn:ogc:def:crs:EPSG::3857</wfs:OtherCRS>
      <ows:WGS84BoundingBox>
        <ows:LowerCorner>-125 25</ows:LowerCorner>
        <ows:UpperCorner>-66 49</ows:UpperCorner>
      </ows:WGS84BoundingBox>
    </wfs:FeatureType>
    <wfs:FeatureType>
      <wfs:Name>hydro:watersheds</wfs:Name>
      <wfs:Title>Watershed Boundaries</wfs:Title>
      <wfs:DefaultCRS>urn:ogc:def:crs:EPSG::4326</wfs:DefaultCRS>
      <ows:WGS84BoundingBox>
        <ows:LowerCorner>-180 -90</ows:LowerCorner>
        <ows:UpperCorner>180 90</ows:UpperCorner>
      </ows:WGS84BoundingBox>
    </wfs:FeatureType>
  </wfs:FeatureTypeList>
</wfs:WFS_Capabilities>
'''

wfs = WebFeatureService('https://example.com/wfs', version='2.0.0', xml=WFS_CAPABILITIES)
print(f"WFS version: {wfs.version}, title: {wfs.identification.title}")
print(f"Feature types: {list(wfs.contents)}")
assert wfs.identification.title == 'Example Hydrology Feature Service'
assert list(wfs.contents) == ['hydro:water_stations', 'hydro:watersheds']
assert [op.name for op in wfs.operations] == ['GetCapabilities', 'DescribeFeatureType', 'GetFeature']

stations = wfs.contents['hydro:water_stations']
print(f"water_stations: title={stations.title!r}, bbox={stations.boundingBoxWGS84}")
assert stations.title == 'Water Monitoring Stations'
assert stations.boundingBoxWGS84 == (-125.0, 25.0, -66.0, 49.0)
assert len(stations.crsOptions) == 2

# Build (but do not send) a GetFeature request URL
url = wfs.getGETGetFeatureRequest(typename=['hydro:water_stations'], maxfeatures=100,
                                  outputFormat='application/json')
query = parse_qs(urlparse(url).query)
print(f"GetFeature URL: {url}")
assert url.startswith('https://example.com/wfs?')
assert query['service'] == ['WFS'] and query['request'] == ['GetFeature']
assert query['typenames'] == ['hydro:water_stations']
assert query['count'] == ['100']
assert query['outputFormat'] == ['application/json']

print("Example 2 finished")

## Example 3: WCS coverage metadata

In [ ]:
# Example 3: WCS (Web Coverage Service) client on an embedded 1.0.0 GetCapabilities document
from owslib.wcs import WebCoverageService

WCS_CAPABILITIES = b'''<?xml version="1.0" encoding="UTF-8"?>
<WCS_Capabilities version="1.0.0" xmlns="http://www.opengis.net/wcs"
    xmlns:gml="http://www.opengis.net/gml" xmlns:xlink="http://www.w3.org/1999/xlink">
  <Service>
    <name>WCS</name>
    <label>Example Elevation Coverage Service</label>
    <description>Offline capabilities document for a smoke test.</description>
    <keywords><keyword>elevation</keyword><keyword>DEM</keyword></keywords>
    <fees>NONE</fees>
    <accessConstraints>NONE</accessConstraints>
  </Service>
  <Capability>
    <Request>
      <GetCapabilities><DCPType><HTTP><Get><OnlineResource xlink:href="https://example.com/wcs?"/></Get></HTTP></DCPType></GetCapabilities>
      <DescribeCoverage><DCPType><HTTP><Get><OnlineResource xlink:href="https://example.com/wcs?"/></Get></HTTP></DCPType></DescribeCoverage>
      <GetCoverage><DCPType><HTTP><Get><OnlineResource xlink:href="https://example.com/wcs?"/></Get></HTTP></DCPType></GetCoverage>
    </Request>
    <Exception><Format>application/vnd.ogc.se_xml</Format></Exception>
  </Capability>
  <ContentMetadata>
    <CoverageOfferingBrief>
      <name>elevation</name>
      <label>Digital Elevation Model</label>
      <description>Height above sea level in meters.</description>
      <lonLatEnvelope srsName="urn:ogc:def:crs:OGC:1.3:CRS84">
        <gml:pos>-180 -90</gml:pos>
        <gml:pos>180 90</gml:pos>
      </lonLatEnvelope>
      <keywords><keyword>elevation</keyword></keywords>
    </CoverageOfferingBrief>
    <CoverageOfferingBrief>
      <name>temperature</name>
      <label>Land Surface Temperature</label>
      <lonLatEnvelope srsName="urn:ogc:def:crs:OGC:1.3:CRS84">
        <gml:pos>-125 25</gml:pos>
        <gml:pos>-66 49</gml:pos>
      </lonLatEnvelope>
    </CoverageOfferingBrief>
  </ContentMetadata>
</WCS_Capabilities>
'''

wcs = WebCoverageService('https://example.com/wcs', version='1.0.0', xml=WCS_CAPABILITIES)
print(f"WCS version: {wcs.version}, title: {wcs.identification.title}")
print(f"Coverages: {list(wcs.contents)}")
assert wcs.version == '1.0.0'
assert wcs.identification.title == 'Example Elevation Coverage Service'
assert wcs.identification.keywords == ['elevation', 'DEM']
assert list(wcs.contents) == ['elevation', 'temperature']
assert [op.name for op in wcs.operations] == ['GetCapabilities', 'DescribeCoverage', 'GetCoverage']

elevation = wcs['elevation']
print(f"elevation: title={elevation.title!r}, bbox={elevation.boundingBoxWGS84}")
assert elevation.title == 'Digital Elevation Model'
assert elevation.boundingBoxWGS84 == (-180.0, -90.0, 180.0, 90.0)
assert wcs['temperature'].boundingBoxWGS84 == (-125.0, 25.0, -66.0, 49.0)

print("Example 3 finished")

## Example 4: CSW records and filter encoding

In [ ]:
# Example 4: parse a CSW (Catalogue Service for the Web) Dublin Core record and build
# OGC Filter Encoding constraints with owslib.fes
from owslib import fes
from owslib.csw import CswRecord
from owslib.etree import etree

RECORD = b'''<csw:Record xmlns:csw="http://www.opengis.net/cat/csw/2.0.2"
    xmlns:dc="http://purl.org/dc/elements/1.1/" xmlns:dct="http://purl.org/dc/terms/">
  <dc:identifier>12345</dc:identifier>
  <dc:title>Global Ocean Temperature Analysis</dc:title>
  <dc:type>dataset</dc:type>
  <dc:subject>oceanography</dc:subject>
  <dc:subject>temperature</dc:subject>
  <dc:format>NetCDF</dc:format>
  <dct:modified>2024-08-15</dct:modified>
  <dct:abstract>Monthly global ocean temperature from satellite observations.</dct:abstract>
  <dct:references scheme="OGC:WMS">https://example.com/wms</dct:references>
</csw:Record>'''

record = CswRecord(etree.fromstring(RECORD))
print(f"Record {record.identifier}: {record.title} ({record.type}, {record.format})")
print(f"Subjects: {record.subjects}, references: {record.references}")
assert record.identifier == '12345'
assert record.title == 'Global Ocean Temperature Analysis'
assert record.subjects == ['oceanography', 'temperature']
assert record.references == [{'scheme': 'OGC:WMS', 'url': 'https://example.com/wms'}]

OGC = '{http://www.opengis.net/ogc}'
state_filter = fes.PropertyIsEqualTo('STATE', 'CA')
element = state_filter.toXML()
assert element.tag == OGC + 'PropertyIsEqualTo'
assert [child.text for child in element] == ['STATE', 'CA']

text_filter = fes.PropertyIsLike('AnyText', '%ocean temperature%')
assert text_filter.toXML().tag == OGC + 'PropertyIsLike'

bbox_filter = fes.BBox([-125, 25, -66, 49])
bbox_element = bbox_filter.toXML()
assert bbox_element.tag == OGC + 'BBOX'
assert bbox_element[1].tag == '{http://www.opengis.net/gml}Envelope'

# A list inside the constraint list means AND; the request is an ogc:Filter document
request = fes.FilterRequest().setConstraintList([[state_filter, bbox_filter]], tostring=True)
print(f"Filter request: {request[:120]}...")
root = etree.fromstring(request)
assert root.tag == OGC + 'Filter'
assert root[0].tag == OGC + 'And'
assert [child.tag for child in root[0]] == [OGC + 'PropertyIsEqualTo', OGC + 'BBOX']

print("Example 4 finished")

## Example 5: OWSLib utility functions

In [ ]:
# Example 5: owslib.util helpers for request URLs, XML values, authentication and errors
from urllib.parse import parse_qs, urlparse

from owslib.etree import etree
from owslib.util import (Authentication, ServiceException, build_get_url,
                         clean_ows_url, testXMLValue, xmltag_split)

# build_get_url keeps existing query parameters and appends the new ones
url = build_get_url('https://example.com/wms?map=demo',
                    {'service': 'WMS', 'version': '1.3.0', 'request': 'GetMap', 'layers': 'radar'})
print(f"GetMap URL: {url}")
query = parse_qs(urlparse(url).query)
assert query == {'map': ['demo'], 'service': ['WMS'], 'version': ['1.3.0'],
                 'request': ['GetMap'], 'layers': ['radar']}

# Existing parameters win unless overwrite=True
assert build_get_url('https://example.com/ows?service=WFS', {'service': 'WMS'}) == 'https://example.com/ows?service=WFS'
assert build_get_url('https://example.com/ows?service=WFS', {'service': 'WMS'}, overwrite=True) == 'https://example.com/ows?service=WMS'

# clean_ows_url strips the OGC request parameters and keeps the others
cleaned = clean_ows_url('https://example.com/wms?service=WMS&request=GetCapabilities&version=1.3.0&map=demo')
print(f"Cleaned URL: {cleaned}")
assert cleaned == 'https://example.com/wms?map=demo'

# XML helpers
element = etree.fromstring(b'<root><value>  radar  </value></root>')
assert testXMLValue(element.find('value')) == 'radar'
assert testXMLValue(element.find('missing')) is None
assert xmltag_split('{http://www.opengis.net/wms}Layer') == 'Layer'

# Authentication settings object and the service exception type
auth = Authentication(username='user', password='pw')
assert auth.username == 'user' and auth.password == 'pw' and auth.verify is True
try:
    raise ServiceException('LayerNotDefined')
except ServiceException as e:
    print(f"ServiceException raised: {e}")
    assert str(e) == 'LayerNotDefined'

print("Example 5 finished")